# Evaluación Sumativa 2 — Taller I
## Procesamiento distribuido de datos históricos de los Juegos Olímpicos con Apache Spark

**Asignatura:** MCDI502 — Gestión de Datos y Tecnologías  
**Grupo:** 3  
**Notebook de entrega:** `mcdi502_s2_g3.ipynb`

### Objetivo
Aplicar RDDs, DataFrames, transformaciones, integración de archivos CSV y JSON, Spark SQL, persistencia, caché y particionamiento dentro de un flujo reproducible.

> **Ejecución:** ejecute las celdas de arriba hacia abajo. En Google Colab use un entorno Python 3 sin GPU.

## Correspondencia con la pauta

| Punto solicitado | Evidencia en el notebook |
|---|---|
| Configuración | Java 17, PySpark 4.0.1, variables de entorno, `SparkSession` y `SparkContext` |
| RDDs | `deportista` con 6 particiones, `deportista2`, `deportistaTotal`, conteo y conversión a DataFrame |
| Transformaciones | `MayorEdad`, `Deportistas_mujer` y conversión a mayúsculas |
| DataFrames | `Evento`, `Resultado`, `Equipos`, `Juego` e integración completa |
| Optimización | AQE, caché, persistencia, broadcast joins y plan de ejecución |
| Paralelismo | DataFrame final con 5 particiones |
| Inspección | Filas, tipos, esquema y muestra |
| Columnas calculadas | `IMC` y `Descripción_sexo` |
| Spark SQL | Medallas por equipo y estadísticas por medalla, temporada y sexo |

# 1. Configuración inicial del entorno

En Google Colab se instala Java 17 y PySpark 4.0.1. Esta versión evita el conflicto observado con el paquete `dataproc-spark-connect` preinstalado en Colab.

En una ejecución local, esta celda puede omitirse después de instalar las dependencias de `requirements.txt`.

In [ ]:
# Detectar Google Colab antes de ejecutar comandos exclusivos de Linux/Colab.
import importlib.util

EN_COLAB = importlib.util.find_spec("google.colab") is not None
print("Ejecución en Google Colab:", EN_COLAB)

if EN_COLAB:
    !apt-get update -qq
    !apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
    !pip uninstall -y -q pyspark py4j > /dev/null 2>&1
    !pip install -q "pyspark[connect]==4.0.1"
    print("Java 17 y PySpark 4.0.1 instalados.")
else:
    print("Entorno local detectado: use las dependencias de requirements.txt.")

In [ ]:
import csv
import os
import sys
import zipfile
from pathlib import Path

import pyspark
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

if EN_COLAB:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "Configurado por el sistema"))
print("SPARK_HOME:", os.environ["SPARK_HOME"])

In [ ]:
# Crear SparkSession y SparkContext.
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("MCDI502_S2_Grupo3_Olimpiadas")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Aplicación:", sc.appName)
print("Versión de Spark:", spark.version)
print("Paralelismo predeterminado:", sc.defaultParallelism)
print("AQE habilitado:", spark.conf.get("spark.sql.adaptive.enabled"))

## 1.1 Carga y localización de los archivos

El notebook busca los archivos dentro de `data/raw`, en el directorio actual y en rutas habituales de Colab. Si no los encuentra, permite subir el ZIP del repositorio o los archivos individuales.

In [ ]:
ALIAS_ARCHIVOS = {
    "deportista": ["deportista.csv"],
    "deportista2": ["deportista2.csv"],
    "evento": ["evento.csv", "eventos.csv"],
    "equipo": ["equipo.csv", "equipos.csv"],
    "resultado": ["resultados.csv", "resultado.csv"],
    "juego": ["juegos.json", "juego.json"],
}

RAICES_BUSQUEDA = [
    Path.cwd(),
    Path.cwd() / "data" / "raw",
    Path("/content"),
    Path("/content/data/raw"),
    Path("/content/GESTION-DE-DATOS-Y-TECNOLOGIAS-completo/data/raw"),
    Path("/content/GESTION-DE-DATOS-Y-TECNOLOGIAS/data/raw"),
]


def buscar_archivo(nombres):
    for raiz in RAICES_BUSQUEDA:
        for nombre in nombres:
            candidato = raiz / nombre
            if candidato.exists():
                return candidato.resolve()
    return None


def resolver_rutas():
    return {clave: buscar_archivo(nombres) for clave, nombres in ALIAS_ARCHIVOS.items()}


RUTAS = resolver_rutas()
faltantes = [clave for clave, ruta in RUTAS.items() if ruta is None]

if faltantes and EN_COLAB:
    print("No se encontraron todos los datos. Suba el ZIP completo o los archivos faltantes.")
    from google.colab import files
    subidos = files.upload()

    for nombre in subidos:
        ruta_subida = Path("/content") / nombre
        if ruta_subida.suffix.lower() == ".zip":
            destino = Path("/content/proyecto_mcdi502")
            destino.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(ruta_subida) as zf:
                zf.extractall(destino)
            RAICES_BUSQUEDA.extend([destino, destino / "data" / "raw"])
            RAICES_BUSQUEDA.extend([p for p in destino.rglob("raw") if p.is_dir()])
        else:
            RAICES_BUSQUEDA.append(Path("/content"))

    RUTAS = resolver_rutas()
    faltantes = [clave for clave, ruta in RUTAS.items() if ruta is None]

if faltantes:
    raise FileNotFoundError(
        "No se encontraron los archivos requeridos: " + ", ".join(faltantes)
    )

print("Archivos localizados:")
for clave, ruta in RUTAS.items():
    print(f"- {clave:12}: {ruta}")

# 2. RDDs: creación y unión (ID 3.1)

Los archivos de deportistas no tienen encabezado. Cada registro se transforma a una tupla con siete campos. Se corrige además una fila irregular que contiene una columna vacía adicional al final.

In [ ]:
def parsear_deportista(linea):
    valores = next(csv.reader([linea]))

    if len(valores) == 8 and valores[-1].strip() == "":
        valores = valores[:-1]

    if len(valores) != 7:
        raise ValueError(f"Registro de deportista inválido: {linea}")

    return (
        int(valores[0]),              # deportista_id
        valores[1].strip(),           # nombre
        int(valores[2]),              # genero
        int(float(valores[3])),       # edad
        float(valores[4]),            # altura
        float(valores[5]),            # peso
        int(valores[6]),              # equipo_id
    )


# RDD solicitado con exactamente 6 particiones.
deportista = (
    sc.textFile(str(RUTAS["deportista"]), minPartitions=6)
    .map(parsear_deportista)
    .repartition(6)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

deportista2 = (
    sc.textFile(str(RUTAS["deportista2"]))
    .map(parsear_deportista)
)

deportistaTotal = deportista.union(deportista2).persist(StorageLevel.MEMORY_AND_DISK)

cantidad_deportistas = deportistaTotal.count()

print("Particiones del RDD deportista:", deportista.getNumPartitions())
print("Particiones del RDD deportista2:", deportista2.getNumPartitions())
print("Registros de deportistaTotal:", cantidad_deportistas)
print("Primeros 5 registros:")
for fila in deportistaTotal.take(5):
    print(fila)

assert deportista.getNumPartitions() == 6, "El RDD deportista debe tener 6 particiones."

## 2.1 Conversión de `deportistaTotal` a DataFrame

La pauta utiliza el mismo nombre `deportista` para el RDD inicial y el DataFrame final. Antes de sobrescribir la variable se conserva una referencia como `deportista_rdd`.

In [ ]:
esquema_deportista = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre", StringType(), False),
    StructField("genero", IntegerType(), False),
    StructField("edad", IntegerType(), True),
    StructField("altura", DoubleType(), True),
    StructField("peso", DoubleType(), True),
    StructField("equipo_id", IntegerType(), True),
])

deportista_rdd = deportista

deportista = (
    spark.createDataFrame(deportistaTotal, schema=esquema_deportista)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

deportista.show(5, truncate=False)
deportista.printSchema()

# 3. RDDs: transformaciones (ID 3.2)

- `MayorEdad`: deportistas con edad igual o superior a 18 años.
- `Deportistas_mujer`: deportistas cuyo código de género es 2.
- `deportistaTotal_mayusculas`: transforma a mayúsculas todos los valores de tipo texto.

In [ ]:
MayorEdad = deportistaTotal.filter(lambda fila: fila[3] >= 18)
Deportistas_mujer = deportistaTotal.filter(lambda fila: fila[2] == 2)

deportistaTotal_mayusculas = deportistaTotal.map(
    lambda fila: tuple(valor.upper() if isinstance(valor, str) else valor for valor in fila)
)

print("Mayores de edad:", MayorEdad.count())
print("Deportistas mujeres:", Deportistas_mujer.count())
print("Ejemplo convertido a mayúsculas:", deportistaTotal_mayusculas.first())

# 4. DataFrames: creación e integración (ID 3.1)

Se crean los DataFrames con los nombres exigidos: `Evento`, `Resultado`, `Equipos` y `Juego`.

- CSV: datos estructurados.
- JSON: datos semiestructurados.
- Se aplican esquemas explícitos para evitar inferencias inconsistentes.

In [ ]:
# -------------------------
# DataFrame Equipos
# -------------------------
esquema_equipos = StructType([
    StructField("id", IntegerType(), False),
    StructField("equipo", StringType(), False),
    StructField("sigla", StringType(), True),
])

Equipos = (
    spark.read
    .option("header", True)
    .schema(esquema_equipos)
    .csv(str(RUTAS["equipo"]))
    .withColumnRenamed("id", "equipo_id")
)

# -------------------------
# DataFrame Evento
# -------------------------
def limpiar_linea_evento(linea):
    texto = linea.strip()
    if texto.startswith('"') and texto.endswith('"'):
        texto = texto[1:-1].replace('""', '"')
    valores = next(csv.reader([texto]))
    if len(valores) != 3:
        raise ValueError(f"Evento inválido: {linea}")
    deporte_id = None if valores[2].strip() in {"", "#N/A"} else int(valores[2])
    return (int(valores[0]), valores[1].strip(), deporte_id)

esquema_evento = StructType([
    StructField("evento_id", IntegerType(), False),
    StructField("evento", StringType(), False),
    StructField("deporte_id", IntegerType(), True),
])

lineas_evento = sc.textFile(str(RUTAS["evento"]))
encabezado_evento = lineas_evento.first()
Evento = spark.createDataFrame(
    lineas_evento.filter(lambda linea: linea != encabezado_evento).map(limpiar_linea_evento),
    esquema_evento,
)

# -------------------------
# DataFrame Resultado
# -------------------------
esquema_resultado = StructType([
    StructField("resultado_id", IntegerType(), False),
    StructField("medalla", StringType(), True),
    StructField("deportista_id", IntegerType(), False),
    StructField("juego_id", IntegerType(), False),
    StructField("evento_id", IntegerType(), True),
])

Resultado = (
    spark.read
    .option("header", True)
    .option("delimiter", ";")
    .option("nullValue", "#N/A")
    .schema(esquema_resultado)
    .csv(str(RUTAS["resultado"]))
)

# -------------------------
# DataFrame Juego
# -------------------------
esquema_juego = StructType([
    StructField("juego_id", IntegerType(), False),
    StructField("ano", StringType(), True),
    StructField("temporada", IntegerType(), True),
    StructField("ciudad", StringType(), True),
])

Juego_original = (
    spark.read
    .option("multiLine", True)
    .schema(esquema_juego)
    .json(str(RUTAS["juego"]))
)

# El archivo fuente usa 'temporada' para el año y 'ciudad' para Verano/Invierno.
# Se normaliza para que la agregación solicitada por temporada sea correcta.
Juego = Juego_original.select(
    "juego_id",
    F.col("ano").alias("edicion"),
    F.col("temporada").alias("anio"),
    F.col("ciudad").alias("temporada"),
)

print("Equipos:", Equipos.count())
print("Eventos:", Evento.count())
print("Resultados:", Resultado.count())
print("Juegos:", Juego.count())

## 4.1 Optimización básica de los DataFrames

Se usan las técnicas solicitadas por el indicador 3.4:

1. **Persistencia y caché** para evitar recalcular tablas reutilizadas.
2. **AQE** para adaptar el plan según estadísticas de ejecución.
3. **Broadcast joins** para dimensiones pequeñas (`Equipos`, `Evento` y `Juego`).
4. **Materialización** mediante `count()` para cargar la caché.

In [ ]:
deportista.persist(StorageLevel.MEMORY_AND_DISK)
Resultado.persist(StorageLevel.MEMORY_AND_DISK)
Equipos.cache()
Evento.cache()
Juego.cache()

for nombre, dataframe in {
    "deportista": deportista,
    "Resultado": Resultado,
    "Equipos": Equipos,
    "Evento": Evento,
    "Juego": Juego,
}.items():
    print(f"{nombre:12}: {dataframe.count():,} registros")

## 4.2 Integración de todos los DataFrames

`Resultado` funciona como tabla central y se relaciona con deportistas, equipos, eventos y juegos. Se utilizan uniones izquierdas para no perder resultados que contengan referencias faltantes o inválidas.

In [ ]:
datos_integrados = (
    Resultado.alias("r")
    .join(deportista.alias("d"), on="deportista_id", how="left")
    .join(F.broadcast(Equipos).alias("e"), on="equipo_id", how="left")
    .join(F.broadcast(Evento).alias("ev"), on="evento_id", how="left")
    .join(F.broadcast(Juego).alias("j"), on="juego_id", how="left")
    .select(
        "resultado_id",
        "deportista_id",
        "nombre",
        "genero",
        "edad",
        "altura",
        "peso",
        "equipo_id",
        F.coalesce(F.col("equipo"), F.lit("Equipo no informado")).alias("equipo"),
        "sigla",
        "evento_id",
        "evento",
        "deporte_id",
        "juego_id",
        "edicion",
        "anio",
        "temporada",
        "medalla",
    )
)

print("Registros integrados:", datos_integrados.count())
datos_integrados.show(5, truncate=False)

# 5. Paralelismo (ID 3.4)

El DataFrame resultante se redistribuye explícitamente en **5 particiones**. Se usa `equipo_id` como clave de particionamiento para favorecer las agregaciones posteriores por equipo.

In [ ]:
datos_integrados = (
    datos_integrados
    .repartition(5, "equipo_id")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Materializar la nueva distribución.
datos_integrados.count()

print("Cantidad de particiones:", datos_integrados.rdd.getNumPartitions())
assert datos_integrados.rdd.getNumPartitions() == 5

# 6. Inspección del DataFrame (ID 3.1)

Se muestra la cantidad de filas, los tipos de datos, el esquema y una muestra de registros.

In [ ]:
print("Cantidad de filas:", datos_integrados.count())

print("\nTipos de datos:")
for nombre, tipo in datos_integrados.dtypes:
    print(f"- {nombre}: {tipo}")

print("\nEsquema completo:")
datos_integrados.printSchema()

print("\nMuestra:")
datos_integrados.show(10, truncate=False)

# 7. Columnas calculadas

- **IMC:** peso en kilogramos dividido por la altura en metros al cuadrado.
- **Descripción_sexo:** `Hombre` para sexo 1, `Mujer` para sexo 2 y `No informado` para otros valores.

Los valores cero de edad, altura y peso se interpretan como datos ausentes para no distorsionar las estadísticas.

In [ ]:
datos_integrados = (
    datos_integrados
    .withColumn(
        "IMC",
        F.when(
            (F.col("altura") > 0) & (F.col("peso") > 0),
            F.round(F.col("peso") / F.pow(F.col("altura") / F.lit(100.0), 2), 2),
        ).otherwise(F.lit(None).cast(DoubleType())),
    )
    .withColumn(
        "Descripción_sexo",
        F.when(F.col("genero") == 1, F.lit("Hombre"))
        .when(F.col("genero") == 2, F.lit("Mujer"))
        .otherwise(F.lit("No informado")),
    )
    .withColumn("edad_valida", F.when(F.col("edad") > 0, F.col("edad")))
    .withColumn("altura_valida", F.when(F.col("altura") > 0, F.col("altura")))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

datos_integrados.select(
    "deportista_id", "nombre", "altura", "peso", "IMC", "Descripción_sexo"
).show(10, truncate=False)

# 8. Agregaciones requeridas con Spark SQL (ID 3.3)

Se registra el DataFrame integrado como una vista temporal llamada `olimpicos`. Las consultas se implementan expresamente con Spark SQL.

In [ ]:
datos_integrados.createOrReplaceTempView("olimpicos")
print("Vista temporal 'olimpicos' creada.")

## 8.1 Cantidad de medallas de oro, plata y bronce por equipo

In [ ]:
medallas_por_equipo = spark.sql("""
    SELECT
        equipo,
        SUM(CASE WHEN medalla = 'Gold' THEN 1 ELSE 0 END) AS oro,
        SUM(CASE WHEN medalla = 'Silver' THEN 1 ELSE 0 END) AS plata,
        SUM(CASE WHEN medalla = 'Bronze' THEN 1 ELSE 0 END) AS bronce,
        SUM(CASE WHEN medalla IN ('Gold', 'Silver', 'Bronze') THEN 1 ELSE 0 END) AS total_medallas
    FROM olimpicos
    GROUP BY equipo
    HAVING total_medallas > 0
    ORDER BY total_medallas DESC, equipo
""")

medallas_por_equipo.show(25, truncate=False)

## 8.2 Suma, promedio, máximo y mínimo de edad por tipo de medalla

In [ ]:
estadisticas_edad_medalla = spark.sql("""
    SELECT
        medalla AS tipo_medalla,
        SUM(edad_valida) AS suma_edad,
        ROUND(AVG(edad_valida), 2) AS promedio_edad,
        MAX(edad_valida) AS maximo_edad,
        MIN(edad_valida) AS minimo_edad
    FROM olimpicos
    WHERE medalla IN ('Gold', 'Silver', 'Bronze')
    GROUP BY medalla
    ORDER BY CASE medalla
        WHEN 'Gold' THEN 1
        WHEN 'Silver' THEN 2
        WHEN 'Bronze' THEN 3
        ELSE 4
    END
""")

estadisticas_edad_medalla.show(truncate=False)

## 8.3 DataFrame `temporada`: estadísticas de altura agrupadas por temporada

In [ ]:
temporada = spark.sql("""
    SELECT
        temporada,
        ROUND(SUM(altura_valida), 2) AS suma_altura,
        ROUND(AVG(altura_valida), 2) AS promedio_altura,
        MAX(altura_valida) AS maximo_altura,
        MIN(altura_valida) AS minimo_altura
    FROM olimpicos
    WHERE temporada IS NOT NULL
    GROUP BY temporada
    ORDER BY temporada
""")

temporada.show(truncate=False)

## 8.4 DataFrame `sexo`: estadísticas de edad agrupadas por sexo

In [ ]:
sexo = spark.sql("""
    SELECT
        `Descripción_sexo` AS sexo,
        SUM(edad_valida) AS suma_edad,
        ROUND(AVG(edad_valida), 2) AS promedio_edad,
        MAX(edad_valida) AS maximo_edad,
        MIN(edad_valida) AS minimo_edad
    FROM olimpicos
    GROUP BY `Descripción_sexo`
    ORDER BY sexo
""")

sexo.show(truncate=False)

## 8.5 Revisión del plan de ejecución

`explain(mode="formatted")` permite observar agregaciones distribuidas, intercambios, particiones y los broadcast joins utilizados durante la integración.

In [ ]:
medallas_por_equipo.explain(mode="formatted")

# 9. Validaciones finales

Estas comprobaciones verifican que el notebook produjo las estructuras obligatorias y que los resultados tienen contenido.

In [ ]:
columnas_obligatorias = {
    "deportista_id", "nombre", "genero", "edad", "altura", "peso",
    "equipo_id", "evento_id", "juego_id", "medalla", "IMC", "Descripción_sexo",
}

faltan_columnas = columnas_obligatorias.difference(datos_integrados.columns)
assert not faltan_columnas, f"Faltan columnas: {sorted(faltan_columnas)}"
assert deportista_rdd.getNumPartitions() == 6
assert datos_integrados.rdd.getNumPartitions() == 5
assert cantidad_deportistas == deportista.count()
assert medallas_por_equipo.count() > 0
assert estadisticas_edad_medalla.count() == 3
assert temporada.count() > 0
assert sexo.count() >= 2

print("Validaciones completadas correctamente.")
print("Deportistas:", f"{cantidad_deportistas:,}")
print("Filas integradas:", f"{datos_integrados.count():,}")
print("Particiones RDD deportista:", deportista_rdd.getNumPartitions())
print("Particiones DataFrame integrado:", datos_integrados.rdd.getNumPartitions())

# 10. Conclusiones

1. Los RDDs permitieron controlar la lectura, el particionamiento y la limpieza inicial de registros irregulares.
2. Los DataFrames facilitaron el uso de esquemas explícitos y la integración de datos estructurados y semiestructurados.
3. Spark SQL permitió expresar las agregaciones de forma clara y verificable.
4. La persistencia, la caché, AQE, el reparticionamiento y los broadcast joins reducen recomputaciones y movimientos de datos.
5. Las validaciones finales hacen que el flujo sea reproducible y facilitan detectar errores antes de entregar.

## Liberación opcional de recursos

Ejecute esta celda solamente después de revisar y guardar todos los resultados.

In [ ]:
for dataframe in [deportista, Resultado, Equipos, Evento, Juego, datos_integrados]:
    dataframe.unpersist(blocking=False)

spark.catalog.clearCache()
print("Recursos liberados.")